## imports

In [1]:
from pathlib import Path
import json

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

## Carpeta raíz del proyecto

In [2]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("No pude encontrar la raíz del proyecto.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("WORKING_DIR:", WORKING_DIR)
print("MANIFESTS_DIR:", MANIFESTS_DIR)

PROJECT_ROOT: /mnt/d/Universidad/analitica/proyecto_analitica2
DATA_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data
WORKING_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/working
MANIFESTS_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests


## manifiesto y subset large


In [3]:
manifest_path = MANIFESTS_DIR / "manifest_isic_master.csv"
subset_path = WORKING_DIR / "isic" / "subsets" / "isic_subset_large.csv"

manifest_df = pd.read_csv(manifest_path)
subset_df = pd.read_csv(subset_path)

print("manifest_df:", manifest_df.shape)
print("subset_df:", subset_df.shape)

print("\nConteo por split:")
print(subset_df["split_final"].value_counts())

subset_df.head()

manifest_df: (25331, 23)
subset_df: (13000, 23)

Conteo por split:
split_final
train    10000
val       1500
test      1500
Name: count, dtype: int64


,image_name,file_path,target_label,labels_list,label_count,age_approx,anatom_site_general,lesion_id,sex,dataset_name,...,NV,BCC,AK,BKL,DF,VASC,SCC,UNK,group_id,split_final
0,ISIC_0067679,/mnt/d/Universidad/analitica/proyecto_analitic...,BCC,['BCC'],1,60.0,anterior torso,BCN_0001624,male,ISIC_2019,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,BCN_0001624,train
1,ISIC_0033597,/mnt/d/Universidad/analitica/proyecto_analitic...,NV,['NV'],1,NaN,NaN,HAM_0005439,NaN,ISIC_2019,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,HAM_0005439,train
2,ISIC_0025825,/mnt/d/Universidad/analitica/proyecto_analitic...,AK,['AK'],1,80.0,head/neck,HAM_0002232,female,ISIC_2019,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,HAM_0002232,train
3,ISIC_0010573,/mnt/d/Universidad/analitica/proyecto_analitic...,NV,['NV'],1,30.0,posterior torso,NaN,male,ISIC_2019,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ISIC_0010573,train
4,ISIC_0059979,/mnt/d/Universidad/analitica/proyecto_analitic...,NV,['NV'],1,35.0,anterior torso,BCN_0000789,male,ISIC_2019,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BCN_0000789,train


## definir clases y mapa de etiquetas

In [ ]:
class_names = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC", "UNK"]

label_to_idx = {label: i for i, label in enumerate(class_names)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

print("Clases:")
print(class_names)
print("\nNúmero de clases:", len(class_names))

## guardar label map

In [ ]:
label_map_dir = WORKING_DIR / "isic" / "meta"
label_map_dir.mkdir(parents=True, exist_ok=True)

label_map_path = label_map_dir / "isic_label_to_idx.json"
with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump(label_to_idx, f, ensure_ascii=False, indent=2)

print("Label map guardado en:", label_map_path)

## validar target_label

In [ ]:
print("Distribución de target_label en subset:")
print(subset_df["target_label"].value_counts())

invalid_labels = subset_df[~subset_df["target_label"].isin(class_names)]
print("\nFilas con labels inválidas:", len(invalid_labels))

## separar train / val / test

In [ ]:
train_df = subset_df[subset_df["split_final"] == "train"].copy()
val_df = subset_df[subset_df["split_final"] == "val"].copy()
test_df = subset_df[subset_df["split_final"] == "test"].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

## rutas faltantes

In [ ]:
def count_missing_files(df):
    return (~df["file_path"].apply(lambda x: Path(x).exists())).sum()

print("Faltantes en train:", count_missing_files(train_df))
print("Faltantes en val:", count_missing_files(val_df))
print("Faltantes en test:", count_missing_files(test_df))

## transforms básicos

In [ ]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

## Dataset de ISIC

In [ ]:
class ISICDataset(Dataset):
    def __init__(self, dataframe, label_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True).copy()
        self.label_to_idx = label_to_idx
        self.transform = transform

        required_cols = ["file_path", "target_label", "split_final", "image_name"]
        for col in required_cols:
            if col not in self.df.columns:
                raise ValueError(f"Falta la columna requerida: {col}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["file_path"])
        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        target_label = row["target_label"]
        target_idx = self.label_to_idx[target_label]

        return {
            "image": image,
            "target": torch.tensor(target_idx, dtype=torch.long),
            "image_name": row["image_name"],
            "file_path": str(image_path),
            "target_label": target_label,
            "split_final": row["split_final"],
        }

## Creacion de dataset

In [ ]:
train_dataset = ISICDataset(train_df, label_to_idx, transform=train_transform)
val_dataset = ISICDataset(val_df, label_to_idx, transform=eval_transform)
test_dataset = ISICDataset(test_df, label_to_idx, transform=eval_transform)

print("Tamaños de datasets:")
print("train:", len(train_dataset))
print("val:", len(val_dataset))
print("test:", len(test_dataset))

## Dataloaders

In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("Dataloaders creados correctamente.")

## inspección de una muestra

In [ ]:
sample = train_dataset[0]

print("Claves:", sample.keys())
print("Shape imagen:", sample["image"].shape)
print("Target idx:", sample["target"])
print("Target label:", sample["target_label"])
print("image_name:", sample["image_name"])
print("split_final:", sample["split_final"])

## inspección de un batch

In [ ]:
batch = next(iter(train_loader))

print("Tipo de batch:", type(batch))
print("Shape batch imágenes:", batch["image"].shape)
print("Shape batch targets:", batch["target"].shape)
print("Primeros target labels:", batch["target_label"][:5])
print("Primeros image_name:", batch["image_name"][:5])